In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('movies_metadata.csv')

In [3]:
df.head(2)

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0


In [4]:
df.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='object')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  object 
 1   belongs_to_collection  4494 non-null   object 
 2   budget                 45466 non-null  object 
 3   genres                 45466 non-null  object 
 4   homepage               7782 non-null   object 
 5   id                     45466 non-null  object 
 6   imdb_id                45449 non-null  object 
 7   original_language      45455 non-null  object 
 8   original_title         45466 non-null  object 
 9   overview               44512 non-null  object 
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  object 
 12  production_companies   45463 non-null  object 
 13  production_countries   45463 non-null  object 
 14  release_date           45379 non-null  object 
 15  re

In [6]:
df.shape

(45466, 24)

In [7]:
df.isnull().sum()

,0
adult,0
belongs_to_collection,40972
budget,0
genres,0
homepage,37684
id,0
imdb_id,17
original_language,11
original_title,0
overview,954


In [8]:
df = df.drop_duplicates().reset_index(drop = True)

In [9]:
df.shape

(45453, 24)

In [10]:
df = df[['title', 'overview', 'genres', 'tagline', 'vote_average', 'popularity']]

In [11]:
df = df.dropna(subset=['title'])

In [12]:
df['overview'] = df['overview'].fillna(" ")
df['tagline'] = df['tagline'].fillna(" ")

In [13]:
df.iloc[0]['genres']

"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]"

In [14]:
import ast
df['genres']=df['genres'].apply(lambda x:" ".join([i['name'] for i in ast.literal_eval(x)]))

In [15]:
df.isnull().sum()

,0
title,0
overview,0
genres,0
tagline,0
vote_average,0
popularity,0


In [16]:
df['tags']=df['overview']+" "+df['genres']+" "+df['tagline']

In [17]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

In [18]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [19]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [20]:
def preprocessing(text):
  #lowercase
  text = str(text).lower()
  #remove the punctuation
  text = re.sub(r'[^a-zA-Z\s]', '',text)
  #tokenization
  words = text.split()
  #remove stopwords
  words = [word for word in words if word not in stop_words]
  #lemmatization
  words = [lemmatizer.lemmatize(word) for word in words]

  return " ".join(words)


In [21]:
df['tags'] = df['tags'].apply(preprocessing)

In [22]:
df['tags'][1]

'sibling judy peter discover enchanted board game open door magical world unwittingly invite alan adult who trapped inside game year living room alans hope freedom finish game prof risky three find running giant rhinoceros evil monkey terrifying creature adventure fantasy family roll dice unleash excitement'

In [23]:
df = df.reset_index(drop = True)

In [24]:
indices = pd.Series(df.index,index = df['title']).drop_duplicates()
indices

,0
title,
Toy Story,0
Jumanji,1
Grumpier Old Men,2
Waiting to Exhale,3
Father of the Bride Part II,4
...,...
Subdue,45442
Century of Birthing,45443
Betrayal,45444


In [25]:
#vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2), stop_words='english')


In [26]:
tfidf_matrix = tfidf.fit_transform(df['tags'])

In [27]:
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1548114 stored elements and shape (45447, 50000)>

In [28]:
#creating model
from sklearn.metrics.pairwise import cosine_similarity


In [35]:
def recommend(title, n=10):
  if title not in indices:
    return ['movie not found']
  idx = indices[title]
  sim_score = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
  similar_idx = sim_score.argsort()[::-1][1:n+1]
  return df['title'].iloc[similar_idx]

In [36]:
recommend('Toy Story')

,title
2996,Toy Story 2
15344,Toy Story 3
24512,Small Fry
28972,Superstar Goofy
17184,Group Sex
6434,"What's Up, Tiger Lily?"
11396,For Your Consideration
1071,Rebel Without a Cause
1931,Condorman
485,Malice


In [37]:
import pickle
pickle.dump(tfidf_matrix,open('tfidf_matrix.pkl','wb'))
pickle.dump(indices,open('indices.pkl','wb'))
df.to_pickle('df.pkl')
pickle.dump(tfidf,open('tfidf.pkl','wb'))


In [42]:
from google.colab import files

files.download('movies.ipynb')

FileNotFoundError: Cannot find file: movies.ipynb

In [43]:
!ls -F

df.pkl	     movies_metadata.csv  tfidf_matrix.pkl
indices.pkl  sample_data/	  tfidf.pkl
